In [ ]:
import os, pickle, json, warnings, glob
import numpy as np
from scipy.stats import chi2
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForTokenClassification, AutoModel
from torchcrf import CRF
from seqeval.metrics import f1_score
warnings.filterwarnings('ignore')

os.environ['CUDA_VISIBLE_DEVICES'] = '1'
DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BASE     = '/home/misel/stroke_ner/outputs/v2'
TEST_TSV = '/home/misel/stroke_ner/data/test_fixed.tsv'
SAVE_DIR = '/home/misel/stroke_ner/outputs/significance_test'
os.makedirs(SAVE_DIR, exist_ok=True)

LABELS = ['O','B-SYM','I-SYM','B-DGN','I-DGN','B-TMP','I-TMP','B-RF','I-RF']
I2L    = {i:l for i,l in enumerate(LABELS)}
NL     = len(LABELS)

print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print('Ready')

Device: cuda
GPU: NVIDIA RTX A4000
Ready


In [ ]:
def load_tsv(path):
    sentences = []
    toks, labs = [], []
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line == '':
                if toks: sentences.append((toks, labs))
                toks, labs = [], []
            else:
                parts = line.split('\t')
                if len(parts) == 2:
                    toks.append(parts[0])
                    labs.append(parts[1])
    if toks: sentences.append((toks, labs))
    return sentences

test_data = load_tsv(TEST_TSV)
GOLDS     = [labs for _, labs in test_data]
print(f'Test sequences: {len(test_data)}')

Test sequences: 275


In [ ]:
def infer_hf(model_dir, max_len, label):
    tok   = AutoTokenizer.from_pretrained(f'{model_dir}/model')
    model = AutoModelForTokenClassification.from_pretrained(
                f'{model_dir}/model').to(DEVICE)
    model.eval()
    preds = []
    with torch.no_grad():
        for tokens, labels in test_data:
            enc      = tok(tokens, is_split_into_words=True,
                           max_length=max_len, padding='max_length',
                           truncation=True, return_tensors='pt')
            word_ids = enc.word_ids(batch_index=0)
            raw      = torch.argmax(
                model(**{k:v.to(DEVICE) for k,v in enc.items()}).logits,
                dim=-1)[0].cpu().tolist()
            seq, prev = [], None
            for i, wid in enumerate(word_ids):
                if wid is None or wid == prev: continue
                seq.append(I2L[raw[i]]); prev = wid
            gl = len(labels)
            preds.append(seq[:gl] if len(seq)>=gl else seq+['O']*(gl-len(seq)))
    f1 = f1_score(GOLDS, preds, average='macro', zero_division=0)
    print(f'  {label}: F1={f1:.4f}')
    del model; torch.cuda.empty_cache()
    return preds

print('Loading M1...')
m1_preds = infer_hf(f'{BASE}/m1_indobert_linear', 128, 'M1 IndoBERT-base+Lin')

Loading M1...
  M1 IndoBERT-base+Lin: F1=0.7367


In [ ]:
class BertCRF(nn.Module):
    def __init__(self, model_name, num_labels, dropout=0.1):
        super().__init__()
        self.bert    = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(dropout)
        self.linear  = nn.Linear(self.bert.config.hidden_size, num_labels)
        self.crf     = CRF(num_labels, batch_first=True)
    def forward(self, input_ids, attention_mask):
        h = self.bert(input_ids=input_ids,
                      attention_mask=attention_mask).last_hidden_state
        return self.crf.decode(self.linear(self.dropout(h)),
                               mask=attention_mask.bool())

M2_DIR   = f'{BASE}/m2_indobert_crf'
tok_m2   = AutoTokenizer.from_pretrained(f'{M2_DIR}/model')
pt_file  = (glob.glob(f'{M2_DIR}/*.pt') + [f'{M2_DIR}/model.pt'])[0]
model_m2 = BertCRF('indobenchmark/indobert-base-p2', NL).to(DEVICE)
model_m2.load_state_dict(torch.load(pt_file, map_location=DEVICE))
model_m2.eval()

m2_preds = []
with torch.no_grad():
    for tokens, labels in test_data:
        enc  = tok_m2(tokens, is_split_into_words=True,
                      max_length=128, padding='max_length',
                      truncation=True, return_tensors='pt')
        wids = enc.word_ids(batch_index=0)
        out  = model_m2(enc['input_ids'].to(DEVICE),
                        enc['attention_mask'].to(DEVICE))[0]
        seq, prev = [], None
        for i, wid in enumerate(wids):
            if wid is None or wid == prev: continue
            if i < len(out): seq.append(I2L[out[i]])
            prev = wid
        gl = len(labels)
        m2_preds.append(seq[:gl] if len(seq)>=gl else seq+['O']*(gl-len(seq)))

print(f'  M2 CRF: F1={f1_score(GOLDS,m2_preds,average="macro",zero_division=0):.4f}')
del model_m2; torch.cuda.empty_cache()

  M2 CRF: F1=0.7427


In [ ]:
m3_preds = infer_hf(f'{BASE}/m3_xlm_roberta',   64,  'M3 XLM-RoBERTa+Lin')
m4_preds = infer_hf(f'{BASE}/m4_indobert_large', 128, 'M4 IndoBERT-large+Lin')

GPT_PKL = '/home/misel/stroke_ner/outputs/v2/gpt/gpt_results.pkl'
with open(GPT_PKL, 'rb') as f:
    gpt_data = pickle.load(f)
gpt_preds = []
for p, g in zip(gpt_data['preds'], GOLDS):
    if len(p) >= len(g): gpt_preds.append(p[:len(g)])
    else: gpt_preds.append(p + ['O']*(len(g)-len(p)))
print(f'  GPT-5.4: F1={f1_score(GOLDS,gpt_preds,average="macro",zero_division=0):.4f}')

print('\nAll models loaded:')
for name, preds in [('M1',m1_preds),('M2',m2_preds),('M3',m3_preds),
                    ('M4',m4_preds),('GPT',gpt_preds)]:
    print(f'  {name}: {f1_score(GOLDS,preds,average="macro",zero_division=0):.4f}')

  M3 XLM-RoBERTa+Lin: F1=0.7726
  M4 IndoBERT-large+Lin: F1=0.7614
  GPT-5.4: F1=0.6682

All models loaded:
  M1: 0.7367
  M2: 0.7427
  M3: 0.7726
  M4: 0.7614
  GPT: 0.6682


In [ ]:
# ── McNemar's test (span-level) ──────────────────────────────
def extract_spans(seq):
    spans = set()
    start, cur = None, None
    for i, tag in enumerate(seq):
        if tag.startswith('B-'):
            if start is not None: spans.add((start, i, cur))
            start, cur = i, tag[2:]
        elif tag.startswith('I-') and cur == tag[2:]:
            pass
        else:
            if start is not None: spans.add((start, i, cur))
            start, cur = None, None
    if start is not None: spans.add((start, len(seq), cur))
    return spans

def mcnemar_ner(preds_a, preds_b, golds):
    n00 = n01 = n10 = n11 = 0
    for pa, pb, gold in zip(preds_a, preds_b, golds):
        gs  = extract_spans(gold)
        asp = extract_spans(pa)
        bsp = extract_spans(pb)
        for span in gs:
            ac = span in asp
            bc = span in bsp
            if     ac and     bc: n11 += 1
            if     ac and not bc: n10 += 1
            if not ac and     bc: n01 += 1
            if not ac and not bc: n00 += 1
    if n01 + n10 == 0:
        return {'n00':n00,'n01':n01,'n10':n10,'n11':n11,
                'chi2':0.0,'p':1.0,'sig_05':False,'sig_01':False,'sig_001':False}
    chi2_stat = (abs(n01 - n10) - 1.0)**2 / (n01 + n10)
    p_val     = 1 - chi2.cdf(chi2_stat, df=1)
    return {'n00':n00,'n01':n01,'n10':n10,'n11':n11,
            'chi2':chi2_stat,'p':p_val,
            'sig_05':p_val<0.05,'sig_01':p_val<0.01,'sig_001':p_val<0.001}

comparisons = [
    ('M3 XLM-RoBERTa',      m3_preds, 'GPT-5.4 zero-shot',       gpt_preds),
    ('M3 XLM-RoBERTa',      m3_preds, 'M4 IndoBERT-large',        m4_preds),
    ('M3 XLM-RoBERTa',      m3_preds, 'M2 IndoBERT-base+CRF',     m2_preds),
    ('M4 IndoBERT-large',    m4_preds, 'M2 IndoBERT-base+CRF',     m2_preds),
    ('M2 IndoBERT-base+CRF', m2_preds, 'M1 IndoBERT-base+Linear',  m1_preds),
]

print('Running McNemar test (span-level, Yates correction)...')
results_mcn = {}
for name_a, pa, name_b, pb in comparisons:
    r   = mcnemar_ner(pa, pb, GOLDS)
    key = f'{name_a} vs {name_b}'
    results_mcn[key] = r
    sig = '***' if r['sig_001'] else ('**' if r['sig_01'] else
          ('*' if r['sig_05'] else 'n.s.'))
    f1a = f1_score(GOLDS, pa, average='macro', zero_division=0)
    f1b = f1_score(GOLDS, pb, average='macro', zero_division=0)
    print(f'  {name_a} vs {name_b}')
    print(f'    F1: {f1a:.4f} vs {f1b:.4f}  delta={f1a-f1b:+.4f}')
    print(f'    n11={r["n11"]}  n10={r["n10"]}  n01={r["n01"]}  n00={r["n00"]}')
    print(f'    chi2={r["chi2"]:.4f}  p={r["p"]:.6f}  {sig}')
    print()
print('Done')

Running McNemar test (span-level, Yates correction)...
  M3 XLM-RoBERTa vs GPT-5.4 zero-shot
    F1: 0.7726 vs 0.6682  delta=+0.1044
    n11=681  n10=204  n01=57  n00=153
    chi2=81.6705  p=0.000000  ***

  M3 XLM-RoBERTa vs M4 IndoBERT-large
    F1: 0.7726 vs 0.7614  delta=+0.0112
    n11=805  n10=80  n01=62  n00=148
    chi2=2.0352  p=0.153693  n.s.

  M3 XLM-RoBERTa vs M2 IndoBERT-base+CRF
    F1: 0.7726 vs 0.7427  delta=+0.0298
    n11=795  n10=90  n01=65  n00=145
    chi2=3.7161  p=0.053889  n.s.

  M4 IndoBERT-large vs M2 IndoBERT-base+CRF
    F1: 0.7614 vs 0.7427  delta=+0.0187
    n11=800  n10=67  n01=60  n00=168
    chi2=0.2835  p=0.594439  n.s.

  M2 IndoBERT-base+CRF vs M1 IndoBERT-base+Linear
    F1: 0.7427 vs 0.7367  delta=+0.0060
    n11=821  n10=39  n01=29  n00=206
    chi2=1.1912  p=0.275092  n.s.

Done


In [ ]:
#Final table + save
print('McNEMAR TEST RESULTS (span-level, Yates continuity correction)')
print('StrokeID-NER test set, n=275 sequences')

model_f1 = {
    'M3 XLM-RoBERTa':       f1_score(GOLDS,m3_preds,average='macro',zero_division=0),
    'M4 IndoBERT-large':     f1_score(GOLDS,m4_preds,average='macro',zero_division=0),
    'M2 IndoBERT-base+CRF':  f1_score(GOLDS,m2_preds,average='macro',zero_division=0),
    'M1 IndoBERT-base+Linear':f1_score(GOLDS,m1_preds,average='macro',zero_division=0),
    'GPT-5.4 zero-shot':     f1_score(GOLDS,gpt_preds,average='macro',zero_division=0),
}

print(f'\n  {"Comparison":<44} {"dF1":>7} {"chi2":>8} {"p":>10}  Sig.')
for comp, r in results_mcn.items():
    na, nb_ = comp.split(' vs ')
    delta = model_f1[na] - model_f1[nb_]
    sig   = '***' if r['sig_001'] else ('**' if r['sig_01'] else
            ('*' if r['sig_05'] else 'n.s.'))
    print(f'  {comp:<44} {delta:>+7.4f} {r["chi2"]:>8.3f} {r["p"]:>10.6f}  {sig}')

print()
print('  *** p<0.001  ** p<0.01  * p<0.05  n.s. not significant')

#Per-entity for M3 vs GPT
print('\nPer-entity breakdown: M3 XLM-RoBERTa vs GPT-5.4')
for entity in ['DGN','RF','SYM','TMP']:
    n00=n01=n10=n11=0
    for pa,pb,gold in zip(m3_preds,gpt_preds,GOLDS):
        gs  = {s for s in extract_spans(gold) if s[2]==entity}
        asp = {s for s in extract_spans(pa)   if s[2]==entity}
        bsp = {s for s in extract_spans(pb)   if s[2]==entity}
        for sp in gs:
            ac=sp in asp; bc=sp in bsp
            if ac and bc:     n11+=1
            if ac and not bc: n10+=1
            if not ac and bc: n01+=1
            if not ac and not bc: n00+=1
    if n01+n10>0:
        c2 = (abs(n01-n10)-1.0)**2/(n01+n10)
        pv = 1-chi2.cdf(c2,df=1)
        sg = '***' if pv<0.001 else ('**' if pv<0.01 else ('*' if pv<0.05 else 'n.s.'))
        print(f'  {entity}: n10={n10:3d} n01={n01:3d}  chi2={c2:.3f}  p={pv:.6f}  {sg}')
    else:
        print(f'  {entity}: no discordant pairs')

#Save results
save_out = {}
for k,v in results_mcn.items():
    save_out[k] = {kk:(round(vv,6) if isinstance(vv,float)
                       else int(vv) if isinstance(vv,(int,np.integer))
                       else bool(vv))
                   for kk,vv in v.items()}
with open(f'{SAVE_DIR}/mcnemar_results.json','w') as f:
    json.dump(save_out, f, indent=2)
print(f'\nSaved: {SAVE_DIR}/mcnemar_results.json')

McNEMAR TEST RESULTS (span-level, Yates continuity correction)
StrokeID-NER test set, n=275 sequences

  Comparison                                       dF1     chi2          p  Sig.
  ------------------------------------------------------------------------
  M3 XLM-RoBERTa vs GPT-5.4 zero-shot          +0.1044   81.670   0.000000  ***
  M3 XLM-RoBERTa vs M4 IndoBERT-large          +0.0112    2.035   0.153693  n.s.
  M3 XLM-RoBERTa vs M2 IndoBERT-base+CRF       +0.0298    3.716   0.053889  n.s.
  M4 IndoBERT-large vs M2 IndoBERT-base+CRF    +0.0187    0.283   0.594439  n.s.
  M2 IndoBERT-base+CRF vs M1 IndoBERT-base+Linear +0.0060    1.191   0.275092  n.s.

  *** p<0.001  ** p<0.01  * p<0.05  n.s. not significant

Per-entity breakdown: M3 XLM-RoBERTa vs GPT-5.4
----------------------------------------------------
  DGN: n10= 18 n01= 15  chi2=0.121  p=0.727724  n.s.
  RF: n10= 42 n01=  6  chi2=25.521  p=0.000000  ***
  SYM: n10=125 n01= 27  chi2=61.901  p=0.000000  ***
  TMP: n10= 19 n